# 🕵️ Ontology Detective — Seed

> Build stamp: **2026-06-05 16:54:59**

Populates the **Datapolis_DetectiveEH** KQL database with the 5 noir case
datasets and (re-)creates the `DetectiveEvents` telemetry table.

**Run all cells once after `arcade.install("ontology-detective")`**.

## Step 1 — Resolve KQL endpoint

In [ ]:
import os, json, uuid, requests
from IPython.display import Markdown, display

EH_NAME = "Datapolis_DetectiveEH"
DB_NAME = "Datapolis_DetectiveEH"

try:
    import notebookutils
    WORKSPACE_ID = notebookutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = notebookutils.credentials.getToken
except Exception:
    import mssparkutils
    WORKSPACE_ID = mssparkutils.runtime.context.get("currentWorkspaceId")
    _gettoken    = mssparkutils.credentials.getToken

FAB = "https://api.fabric.microsoft.com/v1"
def _fab(url):
    r = requests.get(url, headers={"Authorization": f"Bearer {_gettoken('pbi')}"}, timeout=60)
    r.raise_for_status(); return r.json()

dbs = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/items?type=KQLDatabase").get("value", [])
target = next((d for d in dbs if d["displayName"] == DB_NAME), None)
if not target:
    raise RuntimeError(f"KQL DB '{DB_NAME}' not found — was arcade.install run?")
DB_ID = target["id"]
info = _fab(f"{FAB}/workspaces/{WORKSPACE_ID}/kqlDatabases/{DB_ID}")
KQL_URI = info["properties"]["queryServiceUri"]
print("Workspace :", WORKSPACE_ID)
print("KQL DB    :", DB_NAME, "id=", DB_ID)
print("Endpoint  :", KQL_URI)

## Step 2 — Helpers (KQL management + query)

In [ ]:
def _kql_mgmt(csl: str):
    tok = _gettoken("kusto")
    r = requests.post(f"{KQL_URI}/v1/rest/mgmt",
                      headers={"Authorization": f"Bearer {tok}",
                               "Content-Type": "application/json"},
                      json={"csl": csl, "db": DB_NAME}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"KQL mgmt {r.status_code}: {r.text[:500]}")
    return r.json()

def _kql_query(csl: str):
    tok = _gettoken("kusto")
    r = requests.post(f"{KQL_URI}/v2/rest/query",
                      headers={"Authorization": f"Bearer {tok}",
                               "Content-Type": "application/json"},
                      json={"csl": csl, "db": DB_NAME}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"KQL query {r.status_code}: {r.text[:500]}")
    return r.json()

## Step 3 — Apply schema (idempotent `.create-merge`)

In [ ]:
SCHEMA = r'''
.create-merge table DetectiveEvents (
    EventId: string, Timestamp: datetime, SessionId: string, PlayerId: string,
    EventType: string, CaseId: string, AccusedPerson: string,
    ValidationResult: string, DurationSeconds: long, Rank: string
)

.alter-merge table DetectiveEvents policy retention softdelete = 90d

.create-merge table Case1_Visits (
    PersonName: string, RoomName: string, EnteredAt: datetime, LeftAt: datetime
)
.create-merge table Case2_CameraEvents (
    GuestName: string, Location: string, SeenAt: datetime
)
.create-merge table Case3_PhoneCalls (
    Caller: string, Callee: string, CalledAt: datetime
)
.create-merge table Case4_Aliases (
    AliasName: string, RealName: string
)
.create-merge table Case4_HotelCheckIns (
    UsedName: string, HotelName: string, CheckedInAt: datetime
)
.create-merge table Case5_BankAccounts (
    PersonName: string, AccountKind: string, OpenedAt: datetime
)
.create-merge table Case5_PolicePatrols (
    PersonName: string, PatrolZone: string, SeenAt: datetime, OnDutyRegister: bool
)
.create-merge table Case5_BurnerCalls (
    Caller: string, Callee: string, CalledAt: datetime
)
'''
for block in [b.strip() for b in SCHEMA.split("\n\n") if b.strip()]:
    _kql_mgmt(block)
    print("  ✓", block.splitlines()[0][:80])
print("Schema applied.")

## Step 4 — Load evidence (idempotent: clears each evidence table first)

Each evidence row is sent as inline CSV via `.ingest inline into table T <| ...`.

In [ ]:
EVIDENCE = r'''
.clear table Case1_Visits data

.ingest inline into table Case1_Visits <|
Mrs. Plum,Kitchen,2026-06-05T13:30:00Z,2026-06-05T14:00:00Z
Mrs. Plum,Living Room,2026-06-05T14:00:00Z,2026-06-05T14:35:00Z
Mrs. Plum,Kitchen,2026-06-05T14:35:00Z,2026-06-05T15:00:00Z
Bob Hollowstone,Living Room,2026-06-05T13:55:00Z,2026-06-05T14:05:00Z
Bob Hollowstone,Kitchen,2026-06-05T14:05:00Z,2026-06-05T14:20:00Z
Bob Hollowstone,Living Room,2026-06-05T14:20:00Z,2026-06-05T14:40:00Z
Alice Greengrass,Living Room,2026-06-05T13:50:00Z,2026-06-05T14:30:00Z
Alice Greengrass,Garden,2026-06-05T14:30:00Z,2026-06-05T14:50:00Z
Mortimer Quill,Garden,2026-06-05T13:55:00Z,2026-06-05T14:25:00Z
Mortimer Quill,Living Room,2026-06-05T14:25:00Z,2026-06-05T14:40:00Z

.clear table Case2_CameraEvents data

.ingest inline into table Case2_CameraEvents <|
Lady Marlowe,Ballroom,2026-06-04T21:00:00Z
Lady Marlowe,Etruscan Hall,2026-06-04T21:15:00Z
Lady Marlowe,Garden Terrace,2026-06-04T21:25:00Z
Lord Pembroke,Foyer,2026-06-04T21:00:00Z
Lord Pembroke,Ballroom,2026-06-04T21:10:00Z
Lord Pembroke,Egyptian Wing,2026-06-04T21:20:00Z
Dr. Faraday,Egyptian Wing,2026-06-04T21:05:00Z
Dr. Faraday,Etruscan Hall,2026-06-04T21:25:00Z
Count Volturino,Foyer,2026-06-04T21:00:00Z
Count Volturino,Ballroom,2026-06-04T21:15:00Z
Count Volturino,Garden Terrace,2026-06-04T21:30:00Z
Miss Crispin,Egyptian Wing,2026-06-04T21:00:00Z
Miss Crispin,Foyer,2026-06-04T21:15:00Z
Miss Crispin,Ballroom,2026-06-04T21:20:00Z
Professor Bell,Etruscan Hall,2026-06-04T21:00:00Z
Professor Bell,Ballroom,2026-06-04T21:13:00Z
Professor Bell,Foyer,2026-06-04T21:22:00Z
Madame Volga,Garden Terrace,2026-06-04T21:00:00Z
Madame Volga,Etruscan Hall,2026-06-04T21:30:00Z
Sir Hamilton,Egyptian Wing,2026-06-04T21:00:00Z
Sir Hamilton,Garden Terrace,2026-06-04T21:15:00Z
Sir Hamilton,Ballroom,2026-06-04T21:25:00Z

.clear table Case3_PhoneCalls data

.ingest inline into table Case3_PhoneCalls <|
Vincenzo Lupara,Senator Carballo,2026-06-03T18:30:00Z
Vincenzo Lupara,Dr. Aconite,2026-06-03T20:00:00Z
Maria Rossi,Senator Carballo,2026-06-03T18:00:00Z
Maria Rossi,Bank of Datapolis,2026-06-03T19:30:00Z
Antonio Bruno,Dr. Aconite,2026-06-03T18:15:00Z
Antonio Bruno,Maria Rossi,2026-06-03T19:45:00Z
Elena Verdi,Senator Carballo,2026-06-03T17:00:00Z
Elena Verdi,Dr. Aconite,2026-06-03T23:30:00Z
Giuseppe Neri,Senator Carballo,2026-06-03T16:00:00Z
Giuseppe Neri,Vincenzo Lupara,2026-06-03T20:30:00Z
Carla Bianchi,Dr. Aconite,2026-06-03T15:00:00Z
Carla Bianchi,Maria Rossi,2026-06-03T22:00:00Z
Senator Carballo,Bank of Datapolis,2026-06-03T17:45:00Z
Dr. Aconite,Pharmacy 7,2026-06-03T19:00:00Z

.clear table Case4_Aliases data

.ingest inline into table Case4_Aliases <|
V. Rodriguez,Ricardo Vega
Mister V,Ricardo Vega
Vega R.,Roberto Vega Junior
R. Roberts,Roberto Vega Junior
R. Vega III,Rafael Vega
Rafa V.,Rafael Vega

.clear table Case4_HotelCheckIns data

.ingest inline into table Case4_HotelCheckIns <|
V. Rodriguez,Quantum,2026-06-01T22:30:00Z
Vega R.,Plaza,2026-06-01T19:00:00Z
R. Roberts,Quantum,2026-05-30T20:00:00Z
R. Vega III,Continental,2026-06-01T23:00:00Z
Rafa V.,Quantum,2026-06-02T03:00:00Z
Mister V,Plaza,2026-06-01T18:00:00Z
Vega R.,Quantum,2026-05-29T21:00:00Z

.clear table Case5_BankAccounts data

.ingest inline into table Case5_BankAccounts <|
Madame Cinquedeo,RelayShell,2026-05-15T10:00:00Z
Hugo Pellegrini,RelayShell,2026-05-20T11:00:00Z
Sven Halberd,RelayShell,2026-05-22T14:30:00Z
Iris Velvetan,Standard,2026-05-25T09:15:00Z
Tomas Krall,RelayShell,2026-05-28T16:00:00Z
Madame Cinquedeo,Standard,2024-01-10T10:00:00Z

.clear table Case5_PolicePatrols data

.ingest inline into table Case5_PolicePatrols <|
Madame Cinquedeo,Bank District,2026-06-05T22:55:00Z,false
Hugo Pellegrini,Bank District,2026-06-05T22:30:00Z,true
Sven Halberd,Harbor,2026-06-05T22:50:00Z,false
Tomas Krall,Bank District,2026-06-05T23:00:00Z,true
Iris Velvetan,Bank District,2026-06-05T22:45:00Z,false
Carla Drago,Bank District,2026-06-05T22:40:00Z,false

.clear table Case5_BurnerCalls data

.ingest inline into table Case5_BurnerCalls <|
Madame Cinquedeo,+39-X-USA-E-GETTA,2026-06-05T23:04:00Z
Hugo Pellegrini,+39-X-USA-E-GETTA,2026-06-05T22:00:00Z
Iris Velvetan,+39-X-USA-E-GETTA,2026-06-05T23:30:00Z
Tomas Krall,Iris Velvetan,2026-06-05T23:05:00Z
Sven Halberd,+39-X-USA-E-GETTA,2026-06-05T23:07:00Z
Carla Drago,+39-X-USA-E-GETTA,2026-06-05T23:08:00Z
'''
for block in [b.strip() for b in EVIDENCE.split('\n\n') if b.strip()]:
    _kql_mgmt(block)
    head = block.splitlines()[0][:90]
    print('  ✓', head)
print('Evidence loaded for all 5 cases.')

## Step 5 — Sanity check (row counts per evidence table)

In [ ]:
counts_kql = r'''
union withsource=Table
    Case1_Visits, Case2_CameraEvents, Case3_PhoneCalls,
    Case4_Aliases, Case4_HotelCheckIns,
    Case5_BankAccounts, Case5_PolicePatrols, Case5_BurnerCalls
| summarize Rows=count() by Table
| order by Table asc
'''
res = _kql_query(counts_kql)
rows = res["Tables"][0]["Rows"]
md_lines = ["| Table | Rows |", "|-------|-----:|"] + [f"| `{r[0]}` | {r[1]} |" for r in rows]
display(Markdown("\n".join(md_lines)))
print("✅ Seeded successfully. Open OntologyDetective_CaseFile next.")